In [40]:
import pandas as pd
import requests
print("set up working")

set up working


In [41]:
price_url = "https://api.energy-charts.info/price"

price_params = {
    "country": "de",
    "start": "2026-01-01",
    "end": "2026-01-07"
}

price_response = requests.get(price_url, params=price_params)



In [42]:
print(price_response.status_code)

price_data = price_response.json()

print(len(price_data["unix_seconds"]))
print(price_data["unix_seconds"][:3])

200
672
[1767222000, 1767222900, 1767223800]


In [43]:
price_df["timestamp"] = pd.to_datetime(
    price_data["unix_seconds"], unit="s", utc=True
).tz_convert("Europe/Berlin")

In [44]:
price_df = pd.DataFrame({
    "timestamp": price_df["timestamp"] ,
    "price": price_data["price"]
})

In [45]:
print(price_df.head())
print(price_df.shape)

                  timestamp  price
0 2026-01-01 00:00:00+01:00  63.64
1 2026-01-01 00:15:00+01:00  61.48
2 2026-01-01 00:30:00+01:00  58.00
3 2026-01-01 00:45:00+01:00  49.99
4 2026-01-01 01:00:00+01:00  60.01
(672, 2)


In [46]:
url = "https://api.energy-charts.info/v2/public_power"

params = {
    "country": "de",
    "start": "2026-01-01",
    "end": "2026-01-07"
}

power_response = requests.get(url, params=params)

print(power_response.status_code)
print(power_response.url)

200
https://api.energy-charts.info/v2/public_power?country=de&start=2026-01-01&end=2026-01-07


In [47]:
power_data = power_response.json()
print(power_data.keys())
print(power_data["series"])
print(power_data["data"][0])

dict_keys(['schema_version', 'endpoint', 'country', 'bidding_zone', 'timezone', 'resolution', 'interval_minutes', 'unit', 'generated_at', 'available_from', 'available_until', 'series', 'data', 'attributes', 'license', 'deprecated'])
[{'id': 'hydro_pumped_storage_consumption', 'name': 'Hydro pumped storage consumption'}, {'id': 'cross_border_electricity_trading', 'name': 'Cross border electricity trading'}, {'id': 'hydro_run_of_river', 'name': 'Hydro Run-of-River'}, {'id': 'biomass', 'name': 'Biomass'}, {'id': 'fossil_brown_coal_lignite', 'name': 'Fossil brown coal / lignite'}, {'id': 'fossil_hard_coal', 'name': 'Fossil hard coal'}, {'id': 'fossil_oil', 'name': 'Fossil oil'}, {'id': 'fossil_coal_derived_gas', 'name': 'Fossil coal-derived gas'}, {'id': 'fossil_gas', 'name': 'Fossil gas'}, {'id': 'geothermal', 'name': 'Geothermal'}, {'id': 'hydro_water_reservoir', 'name': 'Hydro water reservoir'}, {'id': 'hydro_pumped_storage', 'name': 'Hydro pumped storage'}, {'id': 'others', 'name': 'Ot

In [48]:
rows = []

for row in power_data["data"]:
    rows.append({
        "timestamp": row["timestamp"],
        "load": row["values"].get("load"),
        "wind_onshore": row["values"].get("wind_onshore"),
        "solar": row["values"].get("solar")
    })

power_df = pd.DataFrame(rows)

In [59]:
power_df["timestamp"] = pd.to_datetime(power_df["timestamp"]).dt.tz_convert("Europe/Berlin")

In [60]:
print(power_df.head())
print(power_df.shape)

                  timestamp     load  wind_onshore  solar
0 2026-01-01 00:00:00+01:00  46427.4       31462.6    0.0
1 2026-01-01 00:15:00+01:00  46085.1       31849.2    0.0
2 2026-01-01 00:30:00+01:00  46038.6       32711.4    0.0
3 2026-01-01 00:45:00+01:00  45940.4       33195.1    0.0
4 2026-01-01 01:00:00+01:00  45656.0       33481.0    0.0
(672, 4)


In [61]:
print(power_df["timestamp"].min(), power_df["timestamp"].max())
print(price_df["timestamp"].min(), price_df["timestamp"].max())

2026-01-01 00:00:00+01:00 2026-01-07 23:45:00+01:00
2026-01-01 00:00:00+01:00 2026-01-07 23:45:00+01:00


In [62]:
print(power_df["timestamp"].dtype)
print(price_df["timestamp"].dtype)
print(power_df["timestamp"].equals(price_df["timestamp"]))

datetime64[ns, Europe/Berlin]
datetime64[ns, Europe/Berlin]
True


In [63]:
merged_df = power_df.merge(price_df, on="timestamp", how="inner")

print(merged_df.head())
print(merged_df.shape)

                  timestamp     load  wind_onshore  solar  price
0 2026-01-01 00:00:00+01:00  46427.4       31462.6    0.0  63.64
1 2026-01-01 00:15:00+01:00  46085.1       31849.2    0.0  61.48
2 2026-01-01 00:30:00+01:00  46038.6       32711.4    0.0  58.00
3 2026-01-01 00:45:00+01:00  45940.4       33195.1    0.0  49.99
4 2026-01-01 01:00:00+01:00  45656.0       33481.0    0.0  60.01
(672, 5)


In [64]:
print(merged_df.isna().sum())

timestamp       0
load            0
wind_onshore    0
solar           0
price           0
dtype: int64


In [65]:
print(merged_df["timestamp"].duplicated().sum())

0


In [66]:
print(merged_df.dtypes)

timestamp       datetime64[ns, Europe/Berlin]
load                                  float64
wind_onshore                          float64
solar                                 float64
price                                 float64
dtype: object


In [67]:
merged_df.describe()

,load,wind_onshore,solar,price
count,672.000000,672.000000,672.000000,672.000000
mean,57451.848958,23787.980804,1512.923512,86.969256
std,8776.611542,11952.991494,2755.845795,46.228163
min,42387.200000,3676.100000,0.000000,-0.010000
25%,50581.575000,13109.800000,0.000000,74.445000
50%,56811.600000,24423.350000,20.000000,89.920000
75%,64114.125000,35401.425000,1872.600000,111.010000
max,77504.700000,44230.700000,12065.600000,244.390000


In [68]:
merged_df[merged_df["price"] < 0]

,timestamp,load,wind_onshore,solar,price
43,2026-01-01 10:45:00+01:00,48872.1,40445.4,6869.4,-0.01
47,2026-01-01 11:45:00+01:00,51014.8,40804.2,8551.9,-0.01
48,2026-01-01 12:00:00+01:00,51261.9,40313.6,8699.7,-0.01
49,2026-01-01 12:15:00+01:00,51439.7,40573.1,8763.7,-0.01
50,2026-01-01 12:30:00+01:00,51539.7,40663.0,8670.7,-0.01
51,2026-01-01 12:45:00+01:00,51357.1,41015.1,8468.1,-0.01
52,2026-01-01 13:00:00+01:00,50948.5,41036.3,8208.5,-0.01
53,2026-01-01 13:15:00+01:00,50692.2,41577.3,7728.3,-0.01
54,2026-01-01 13:30:00+01:00,50798.6,41738.1,7141.6,-0.01
56,2026-01-01 14:00:00+01:00,51133.6,42713.2,5777.5,-0.01


In [69]:
print((merged_df["price"] < 0).sum())

10


In [71]:
print(merged_df["timestamp"].min())
print(merged_df["timestamp"].max())

2026-01-01 00:00:00+01:00
2026-01-07 23:45:00+01:00
